# Titanic 生存预测分析

## 1. 问题定义

**任务描述：** 预测泰坦尼克乘客是否生存（二分类问题）

**为什么选 F1-score 而不是 Accuracy？**
数据集中约62%的乘客死亡，类别不平衡。若模型全部预测"死亡"，Accuracy 仍有62%，但毫无意义。
F1-score 综合考虑精确率和召回率，对不平衡数据更有区分度。

**评估框架：**
- F1-score（主要指标）
- AUC-ROC（综合评估分类能力）
- 混淆矩阵（直观展示预测分布）

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 从 seaborn 内置数据集加载（无需下载）
df = sns.load_dataset('titanic')
print(f"数据集形状: {df.shape}")
df.head()

## 2. 探索性数据分析（EDA）

### 2.1 缺失值分析

首先了解数据质量，缺失值的处理策略将在特征工程阶段决定。

In [ ]:
# 缺失值分析
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'缺失数量': missing, '缺失比例(%)': missing_pct})
missing_df[missing_df['缺失数量'] > 0].sort_values('缺失比例(%)', ascending=False)

### 2.2 生存率分布分析

观察关键特征与生存率的关系，为后续特征选择提供依据。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 总体生存率
df['survived'].value_counts().plot(kind='bar', ax=axes[0], color=['#e74c3c','#2ecc71'])
axes[0].set_title('总体生存分布
(0=死亡, 1=生存)')
axes[0].set_xlabel('')

# 性别与生存率
df.groupby('sex')['survived'].mean().plot(kind='bar', ax=axes[1], color=['#3498db','#e91e63'])
axes[1].set_title('性别与生存率')
axes[1].set_ylabel('生存率')

# 舱位等级与生存率
df.groupby('pclass')['survived'].mean().plot(kind='bar', ax=axes[2], color=['#f39c12','#95a5a6','#7f8c8d'])
axes[2].set_title('舱位等级与生存率')
axes[2].set_ylabel('生存率')

plt.tight_layout()
plt.savefig('eda_survival.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.3 EDA 关键发现

- **性别影响显著：** 女性生存率约74%，男性约19%（"女士优先"原则体现）
- **舱位等级影响显著：** 一等舱生存率约63%，三等舱约24%（经济地位影响逃生机会）
- **缺失值情况：** `age` 缺失约20%，`deck` 缺失约77%，`embarked` 缺失2条

这些发现直接指导了后续的特征工程决策。

## 3. 特征工程

### 决策依据

- **age 缺失用中位数填充（而非均值）：** 年龄分布右偏（有少数高龄乘客），中位数比均值更稳健，不受极端值影响
- **embarked 缺失仅2条：** 用众数填充，影响极小
- **构造 family_size = sibsp + parch + 1：** 假设独行旅客和大家庭的生存策略不同，值得验证
- **删除 deck（缺失77%）：** 信息量极低，填充会引入大量噪声
- **删除 ticket、name：** 高基数类别特征，直接建模效果差（name 中的称谓可作为改进方向）

In [ ]:
df_model = df.copy()

# 填充缺失值
df_model['age'].fillna(df_model['age'].median(), inplace=True)
df_model['embarked'].fillna(df_model['embarked'].mode()[0], inplace=True)

# 构造新特征
df_model['family_size'] = df_model['sibsp'] + df_model['parch'] + 1
df_model['is_alone'] = (df_model['family_size'] == 1).astype(int)

# 类别编码
df_model['sex_encoded'] = LabelEncoder().fit_transform(df_model['sex'])
df_model['embarked_encoded'] = LabelEncoder().fit_transform(df_model['embarked'])

# 选择特征
features = ['pclass', 'age', 'sibsp', 'parch', 'fare',
            'family_size', 'is_alone', 'sex_encoded', 'embarked_encoded']
X = df_model[features]
y = df_model['survived']

print(f"特征维度: {X.shape}")
print(f"类别分布: {y.value_counts().to_dict()}")

### 3.1 特征相关性分析

通过热力图验证特征与目标变量的相关性，确认特征选择的合理性。

In [ ]:
plt.figure(figsize=(10, 8))
corr = df_model[features + ['survived']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('特征相关性矩阵')
plt.tight_layout()
plt.savefig('feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. 多模型对比

### 模型选型理由

- **逻辑回归：** 基线模型，可解释性强，适合二分类，结果可作为参照
- **随机森林：** 集成方法，对异常值鲁棒，自带特征重要性，不易过拟合
- **XGBoost：** 梯度提升框架，在表格数据竞赛中表现优异，通常优于单模型

**评估策略：** 统一使用5折交叉验证，避免单次数据划分的随机性影响结论。

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    '逻辑回归': LogisticRegression(max_iter=1000, random_state=42),
    '随机森林': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': xgb.XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
}

results = {}
for name, model in models.items():
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring="f1")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    results[name] = {
        'CV F1 (mean)': cv_scores.mean().round(4),
        'CV F1 (std)': cv_scores.std().round(4),
        'Test AUC': auc.round(4)
    }

results_df = pd.DataFrame(results).T
print(results_df)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
results_df['CV F1 (mean)'].plot(
    kind='bar', ax=ax,
    color=['#3498db', '#2ecc71', '#e74c3c'],
    yerr=results_df['CV F1 (std)'], capsize=5
)
ax.set_title('三模型 5折交叉验证 F1 对比')
ax.set_ylabel('F1 Score')
ax.set_ylim(0.5, 0.9)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()